### Use Case: Site Response and Wave Detection at a Strong-Motion Station

Japan operates one of the densest strong-motion networks in the world (K-NET, KiK-net) to characterize ground shaking during earthquakes. Two questions are central for every station:

1. **How does the local ground respond?** Soft soil above stiffer bedrock acts like a resonator: seismic waves are trapped in the layer and make it oscillate at its own *predominant period*. Buildings whose natural period lies close to this ground period experience resonance and are particularly vulnerable. Local site amplification has been discussed as a contributing factor to the severe damage in Mashiki during the 2016 Kumamoto earthquakes [source].
2. **When does a specific wave arrive?** Automatic monitoring systems must locate characteristic wave packets, such as the S-wave arrival, within a continuous record, for example for phase picking or for detecting aftershocks that resemble a known event (template matching).

This block investigates both questions using station KMM006 in Kumamoto Prefecture, located almost directly above the hypocenter of the M4.5 event on 2026-09-26.

### Problem Statement

**Autocorrelation (within the signal):** After the direct S wave, the record shows decaying oscillations (coda) that are expected to contain the resonance of the soil layer below the station. The goal is to identify the dominant recurring period in this part of the signal. Peaks in the autocorrelation at non-zero lag correspond to this period and allow an estimate of the predominant ground period at KMM006. The challenge is that an earthquake record is not truly periodic: it is a transient whose oscillations are only quasi-periodic and decay over time.

**Cross-correlation (segment detection):** A characteristic segment of the record, the high-amplitude S-wave packet, is used as a template. The goal is to locate this segment within the full window. The lag of maximum cross-correlation indicates its arrival time. The challenge is to obtain a distinct and reliable detection despite similar-looking oscillations elsewhere in the record, background noise, and amplitude variations.

| | Autocorrelation | Cross-correlation |
|---|---|---|
| Compares | the signal with shifted copies of itself | a template with the signal |
| Question | Does the signal repeat itself, and with which period? | Where does this pattern occur in the signal? |
| Result | dominant period [s] | position of the segment [s] |
| Domain meaning | soil resonance, relevant for building damage | wave arrival time, relevant for detection |

### Signal Selection

| Property | Value |
|---|---|
| Physical origin | Ground acceleration from the M4.5 earthquake on 2026-09-26, 10:31 (JST), Kumamoto, 10 km depth |
| Station / component | KMM006 (K-NET), NS (horizontal) |
| Unit | gal (= cm/s²) |
| Acquisition | Accelerometer, digitized at 100 Hz (raw counts × scale factor) |
| Window | 14–34 s (pre-event noise, P and S waves, coda) |
| Expected structures | Quasi-periodic oscillations in the coda (soil resonance); one distinct, high-amplitude S-wave packet |

**Choice of component.** The horizontal NS component is used because the S wave dominates horizontal motion, which makes the S-wave packet the most distinct segment in the record (peak ≈ −130 gal). Soil resonance also mainly affects horizontal shear motion, which is the motion most relevant for buildings.

**Choice of window.** The window contains every part needed for both analyses:
- *14–15 s, pre-event noise:* a region where the template should not match, which makes it possible to check whether a detection is specific.
- *≈ 16.5–18 s, S-wave packet:* the segment to be detected.
- *≈ 18–34 s, coda:* the decaying, quasi-periodic oscillations used for the autocorrelation.

The window ends at 34 s, where the signal has decayed close to the noise level.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from knet_io import read_knet

FILE = "data/20260926103100/knet/ascii/KMM0062609261031.NS"
T_START, T_END = 14.0, 34.0   # selected window [s]

acc, fs, hdr = read_knet(FILE)
t = np.arange(len(acc)) / fs
mask = (t >= T_START) & (t <= T_END)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(t[mask], acc[mask], lw=0.8)
ax.axvspan(14, 15, color="gray", alpha=0.2, label="Pre-event noise")
ax.axvspan(16.5, 18, color="tab:red", alpha=0.2, label="S-wave packet (segment)")
ax.axvspan(18, 34, color="tab:green", alpha=0.1, label="Coda (recurring oscillations)")
ax.set(xlabel="Time [s]", ylabel="Acceleration [gal]",
       title=f"{hdr['Station Code']} NS – selected window {T_START:.0f}–{T_END:.0f} s")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

### Why Recurring Structures Are Expected

Earthquake ground motion is a transient, not a periodic signal. Nevertheless, recurring structures are expected in the coda: after the direct S wave has passed, seismic energy keeps reflecting between the free surface and the stiffer bedrock below the station. The soil layer therefore "rings" at its natural frequency, which can be approximated by the quarter-wavelength rule $f_0 = v_S / (4H)$, where $v_S$ is the shear-wave velocity of the soil and $H$ the layer thickness. In the record, this appears as fairly regular oscillations after the S-wave packet, whose amplitude decays over time.

The S-wave packet, in contrast, is a single, distinct event with the highest amplitudes in the record. It occurs once and is therefore a suitable segment for detection.

### Why Correlation Is Appropriate

- **Autocorrelation** compares the signal with time-shifted copies of itself. If the signal repeats with period $T$, it is similar to itself at a lag of $T$, producing a peak in the autocorrelation. Because it uses the entire window rather than individual peaks, it is less sensitive to single irregular oscillations, which is important for a decaying, quasi-periodic signal.
- **Cross-correlation** slides the template along the signal and measures their similarity at each position. The position of maximum similarity marks where the segment occurs. This is the principle of the matched filter, a standard method for detecting known waveforms in noisy signals and the basis of template-matching earthquake detection.
- Both methods operate directly on the recorded time series and require no physical model of the source or the soil, which makes them suitable for routine, automatic analysis at many stations.

### Data Source

National Research Institute for Earth Science and Disaster Resilience (NIED): K-NET, KiK-net. doi:10.17598/NIED.0004